In [1]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [2]:
SALEX = pd.read_csv("lexicons/salex_nrc.csv").set_index('term_str')
SALEX.columns = [col.replace('nrc_','') for col in SALEX.columns]

emo_cols = "anger anticipation disgust fear joy sadness surprise trust sentiment".split()

In [3]:
token_files = glob.glob('data/*_tokens.csv')

records = []

for filepath in token_files:
    # Parse country and year from filename
    filename = os.path.basename(filepath)                  # e.g. United_States_of_America_1992_tokens.csv
    parts = filename.replace('_tokens.csv', '').rsplit('_', 1)
    country = parts[0].replace('_', ' ')                   # "United States of America"
    year = int(parts[1])                                   # 1992

    # Load tokens and join lexicon
    tokens = pd.read_csv(filepath)
    tokens_sa = tokens.join(SALEX, on='term_str', how='left')
    tokens_sa[emo_cols] = tokens_sa[emo_cols].fillna(0)

    # Count total tokens and matched tokens for normalization
    n_total = len(tokens_sa)
    n_matched = tokens_sa[emo_cols].any(axis=1).sum()

    if n_matched == 0:
        continue  # skip if no lexicon matches at all

    # Build one record per document
    record = {
        'country': country,
        'year':    year,
        'n_tokens': n_total,
        'n_matched': n_matched,
        'pct_matched': n_matched / n_total,
    }

    # Normalized emotion scores (proportion of matched tokens)
    for col in emo_cols:
        record[col] = tokens_sa[col].sum() / n_matched

    records.append(record)

SCORES = pd.DataFrame(records)
SCORES = SCORES.sort_values(['country', 'year']).reset_index(drop=True)

print(f"Constitutions scored: {len(SCORES)}")
display(SCORES.head())


Constitutions scored: 192


,country,year,n_tokens,n_matched,pct_matched,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,sentiment
0,Afghanistan,2004,10498,1049,0.099924,0.193518,0.405148,0.052431,0.269781,0.329838,0.121068,0.081030,0.655863,0.485224
1,Albania,2008,13961,1411,0.101067,0.165131,0.404678,0.053863,0.244507,0.280652,0.104181,0.068037,0.666194,0.520198
2,Algeria,2008,10756,1122,0.104314,0.098039,0.449198,0.045455,0.179144,0.322638,0.100713,0.084670,0.736185,0.624777
3,Andorra,1993,8793,913,0.103833,0.097481,0.422782,0.061336,0.205915,0.293538,0.084337,0.081051,0.731654,0.619934
4,Angola,2010,26408,2381,0.090162,0.178076,0.364133,0.061739,0.235195,0.271315,0.107518,0.061739,0.650987,0.551449


In [20]:
VDEM = pd.read_csv('data/vdem/vdem.csv', usecols=[
    'country_name',     
    'year',
    'v2x_polyarchy',      # Electoral Democracy Index (0-1)
    'v2x_libdem',         # Liberal Democracy Index (0-1)
    'v2x_partipdem',      # Participatory Democracy Index (0-1)
])

In [21]:
VDEM.head()

,country_name,year,v2x_polyarchy,v2x_libdem,v2x_partipdem
0,Mexico,1789,0.027,0.044,0.006
1,Mexico,1790,0.027,0.044,0.006
2,Mexico,1791,0.027,0.044,0.006
3,Mexico,1792,0.027,0.044,0.006
4,Mexico,1793,0.027,0.044,0.006


In [22]:
VDEM = VDEM.rename(columns={
    'country_name': 'country',
    'v2x_polyarchy': 'electoral_democracy',
    'v2x_libdem':    'liberal_democracy',
    'v2x_partipdem': 'participatory_democracy',
})

In [23]:
MERGED = SCORES.merge(VDEM, on=['country', 'year'], how='left')

# How many matched?
n_matched_vdem = MERGED['electoral_democracy'].notna().sum()
print(f"Matched with V-Dem: {n_matched_vdem} / {len(MERGED)}")

# If country name mismatches are causing gaps, inspect them:
unmatched = MERGED[MERGED['electoral_democracy'].isna()]['country'].tolist()
print("Unmatched countries:", unmatched[:10])

Matched with V-Dem: 155 / 192
Unmatched countries: ['Andorra', 'Antigua and Barbuda', 'Bahamas', 'Belize', 'Bosnia Herzegovina', 'Brunei', 'Congo', 'Cote DIvoire', 'Czech Republic', 'Dominica']


In [24]:
SCORES[SCORES['country']=='Andorra']

,country,year,n_tokens,n_matched,pct_matched,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,sentiment,country_vdem
3,Andorra,1993,8793,913,0.103833,0.097481,0.422782,0.061336,0.205915,0.293538,0.084337,0.081051,0.731654,0.619934,Andorra


In [25]:
print(VDEM['country'].unique())

['Mexico' 'Suriname' 'Sweden' 'Switzerland' 'Ghana' 'South Africa' 'Japan'
 'Burma/Myanmar' 'Russia' 'Albania' 'Egypt' 'Yemen' 'Colombia' 'Poland'
 'Brazil' 'United States of America' 'Portugal' 'El Salvador'
 'South Yemen' 'Bangladesh' 'Bolivia' 'Haiti' 'Honduras' 'Mali' 'Pakistan'
 'Peru' 'Senegal' 'South Sudan' 'Sudan' 'Vietnam' 'Republic of Vietnam'
 'Afghanistan' 'Argentina' 'Ethiopia' 'India' 'Kenya' 'North Korea'
 'South Korea' 'Kosovo' 'Lebanon' 'Nigeria' 'Philippines' 'Tanzania'
 'Taiwan' 'Thailand' 'Uganda' 'Venezuela' 'Benin' 'Bhutan' 'Burkina Faso'
 'Cambodia' 'Indonesia' 'Mozambique' 'Nepal' 'Nicaragua' 'Niger' 'Zambia'
 'Zimbabwe' 'Guinea' 'Ivory Coast' 'Mauritania' 'Canada' 'Australia'
 'Botswana' 'Burundi' 'Cape Verde' 'Central African Republic' 'Chile'
 'Costa Rica' 'Timor-Leste' 'Ecuador' 'France' 'Germany' 'Guatemala'
 'Iran' 'Iraq' 'Ireland' 'Italy' 'Jordan' 'Latvia' 'Lesotho' 'Liberia'
 'Malawi' 'Maldives' 'Mongolia' 'Morocco' 'Netherlands' 'Panama'
 'Papua New Gui

In [ ]:
name_fixes = {
    'Bosnia and Herzegovina':                   'Bosnia Herzegovina',
    'Brunei':                               'Brunei Darussalam',
    'Congo':                                'Republic of the Congo',
    'Cote DIvoire':                         "Cote d'Ivoire",
    'Czech Republic':                       'Czechia', 
    'Gambia':                               'The Gambia',
    'Macedonia':                            'North Macedonia',
    'Guinea Bissau':                        'Guinea-Bissau',
    'Myanmar':                              'Burma/Myanmar',
    'Surinam':                              'Suriname'
    
}

In [27]:
SCORES['country_vdem'] = SCORES['country'].replace(name_fixes)

MERGED = SCORES.merge(
    VDEM,
    left_on=['country_vdem', 'year'],
    right_on=['country', 'year'],
    how='left',
    suffixes=('', '_vdem')
).drop(columns=['country_vdem'])

print(f"After name fixes — matched: {MERGED['electoral_democracy'].notna().sum()} / {len(MERGED)}")

After name fixes — matched: 156 / 192


In [29]:
unmatched = MERGED[MERGED['electoral_democracy'].isna()]['country'].tolist()
print("Unmatched countries:", unmatched)

Unmatched countries: ['Andorra', 'Antigua and Barbuda', 'Bahamas', 'Belize', 'Bosnia Herzegovina', 'Brunei', 'Cote DIvoire', 'Czech Republic', 'Dominica', 'East Timor', 'Gambia', 'German Federal Republic', 'Grenada', 'Guinea Bissau', 'Kiribati', 'Kyrgyz Republic', 'Liechtenstein', 'Macedonia', 'Marshall Islands', 'Micronesia', 'Monaco', 'Myanmar', 'Nauru', 'Palau', 'Peoples Republic of Korea', 'Republic of Korea', 'Samoa', 'Socialist Republic of Vietnam', 'St Kitts and Nevis', 'St Lucia', 'St Vincent and the Grenadines', 'Surinam', 'Swaziland', 'Tonga', 'Turkey', 'Tuvalu']


In [31]:
vdem_countries = set(VDEM['country'].unique())

print("Checking your unmatched list against V-Dem coverage:")
print("-" * 55)

problem_countries = ['Andorra', 'Antigua and Barbuda', 'Bahamas', 'Belize', 'Bosnia Herzegovina', 'Brunei', 'Cote DIvoire', 'Czech Republic', 'Dominica', 'East Timor', 'Gambia', 'German Federal Republic', 'Grenada', 'Guinea Bissau', 'Kiribati', 'Kyrgyz Republic', 'Liechtenstein', 'Macedonia', 'Marshall Islands', 'Micronesia', 'Monaco', 'Myanmar', 'Nauru', 'Palau', 'Peoples Republic of Korea', 'Republic of Korea', 'Samoa', 'Socialist Republic of Vietnam', 'St Kitts and Nevis', 'St Lucia', 'St Vincent and the Grenadines', 'Surinam', 'Swaziland', 'Tonga', 'Turkey', 'Tuvalu']


for c in problem_countries:
    in_vdem = c in vdem_countries
    # Also do a fuzzy check for close matches
    close = [v for v in vdem_countries if c.lower()[:5] in v.lower()]
    print(f"  {c:<30} In V-Dem: {in_vdem}   Close matches: {close[:3]}")


Checking your unmatched list against V-Dem coverage:
-------------------------------------------------------
  Andorra                        In V-Dem: False   Close matches: []
  Antigua and Barbuda            In V-Dem: False   Close matches: []
  Bahamas                        In V-Dem: False   Close matches: []
  Belize                         In V-Dem: False   Close matches: []
  Bosnia Herzegovina             In V-Dem: False   Close matches: ['Bosnia and Herzegovina']
  Brunei                         In V-Dem: False   Close matches: []
  Cote DIvoire                   In V-Dem: False   Close matches: []
  Czech Republic                 In V-Dem: False   Close matches: ['Czechia']
  Dominica                       In V-Dem: False   Close matches: ['Dominican Republic']
  East Timor                     In V-Dem: False   Close matches: []
  Gambia                         In V-Dem: False   Close matches: ['The Gambia']
  German Federal Republic        In V-Dem: False   Close matches: [

In [39]:
# Use V-Dem where available, fall back to Freedom House for unmatched

SCORES['country_clean'] = SCORES['country'].replace(name_fixes)

# First merge with V-Dem
MERGED = SCORES.merge(
    VDEM[['country', 'year', 'electoral_democracy', 'liberal_democracy', 'participatory_democracy']],
    left_on=['country_clean', 'year'],
    right_on=['country', 'year'],
    how='left',
    suffixes=('', '_vdem')
).drop(columns=['country_vdem'], errors='ignore')

In [40]:
name_fixes = {
    # Confirmed by fuzzy matcher - use exact V-Dem names
    'Bosnia Herzegovina':               'Bosnia and Herzegovina',
    'Czech Republic':                   'Czechia',
    'Gambia':                           'The Gambia',
    'German Federal Republic':          'Germany',
    'Guinea Bissau':                    'Guinea-Bissau',
    'Kyrgyz Republic':                  'Kyrgyzstan',
    'Macedonia':                        'North Macedonia',
    'Myanmar':                          'Burma/Myanmar',
    'Surinam':                          'Suriname',

    # Confirmed mappings not caught by fuzzy matcher
    'East Timor':                       'Timor-Leste',
    'Peoples Republic of Korea':        'North Korea',
    'Republic of Korea':                'South Korea',
    'Socialist Republic of Vietnam':    'Vietnam',
    'St Kitts and Nevis':               'Saint Kitts and Nevis',
    'St Lucia':                         'Saint Lucia',
    'St Vincent and the Grenadines':    'Saint Vincent and the Grenadines',
    'Swaziland':                        'Eswatini',
    'Turkey':                           'Türkiye',
    'Cote DIvoire':                     "Côte d'Ivoire",
    'Brunei':                           'Brunei Darussalam',

    # These are genuinely absent from V-Dem (microstates / Pacific islands)
    # Leave as-is so they show up cleanly in the missing report
    # and get picked up by Freedom House fallback:
    # Andorra, Antigua and Barbuda, Bahamas, Belize, Dominica,
    # Grenada, Kiribati, Liechtenstein, Marshall Islands,
    # Micronesia, Monaco, Nauru, Palau, Samoa, Tonga, Tuvalu
}


matched = MERGED[MERGED['electoral_democracy'].notna()]['country'].tolist()
unmatched = MERGED[MERGED['electoral_democracy'].isna()]['country'].tolist()

print(f"Matched:   {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
print("\nUnmatched (should now only be genuine V-Dem gaps):")
for c in sorted(unmatched):
    print(f"  {c}")

spot_checks = [
    'Bosnia Herzegovina', 'Czech Republic', 'Gambia', 
    'Republic of Korea', 'Peoples Republic of Korea',
    'Myanmar', 'Turkey', 'Swaziland'
]

print("\nSpot check - country_clean values:")
print("-" * 50)
check_df = SCORES[SCORES['country'].isin(spot_checks)][['country', 'country_clean', 'year']]
print(check_df.to_string(index=False))

Matched:   170
Unmatched: 22

Unmatched (should now only be genuine V-Dem gaps):
  Andorra
  Antigua and Barbuda
  Bahamas
  Belize
  Brunei
  Congo
  Cote DIvoire
  Dominica
  Grenada
  Kiribati
  Liechtenstein
  Marshall Islands
  Micronesia
  Monaco
  Nauru
  Palau
  Samoa
  St Kitts and Nevis
  St Lucia
  St Vincent and the Grenadines
  Tonga
  Tuvalu

Spot check - country_clean values:
--------------------------------------------------
                  country          country_clean  year
       Bosnia Herzegovina Bosnia and Herzegovina  2009
           Czech Republic                Czechia  2002
                   Gambia             The Gambia  2004
                  Myanmar          Burma/Myanmar  2008
Peoples Republic of Korea            North Korea  1998
        Republic of Korea            South Korea  1987
                Swaziland               Eswatini  2005
                   Turkey                Türkiye  2002


In [41]:
matched_mask = MERGED['electoral_democracy'].notna()

DROPPED = MERGED[~matched_mask][['country', 'year']].copy()
ANALYSIS = MERGED[matched_mask].copy().reset_index(drop=True)

print(f"Countries retained:  {len(ANALYSIS)}")
print(f"Countries dropped:   {len(DROPPED)}")
print(f"\nDropped countries:")
for _, row in DROPPED.iterrows():
    print(f"  {row['country']} ({row['year']})")

Countries retained:  170
Countries dropped:   22

Dropped countries:
  Andorra (1993)
  Antigua and Barbuda (1981)
  Bahamas (2002)
  Belize (2001)
  Brunei (1984)
  Congo (2001)
  Cote DIvoire (2009)
  Dominica (1984)
  Grenada (1992)
  Kiribati (1995)
  Liechtenstein (2003)
  Marshall Islands (1995)
  Micronesia (1990)
  Monaco (2002)
  Nauru (1968)
  Palau (1992)
  Samoa (2010)
  St Kitts and Nevis (1983)
  St Lucia (1978)
  St Vincent and the Grenadines (1979)
  Tonga (1988)
  Tuvalu (1986)


In [42]:
MERGED.head()

,country,year,n_tokens,n_matched,pct_matched,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,sentiment,country_clean,electoral_democracy,liberal_democracy,participatory_democracy
0,Afghanistan,2004,10498,1049,0.099924,0.193518,0.405148,0.052431,0.269781,0.329838,0.121068,0.081030,0.655863,0.485224,Afghanistan,0.235,0.089,0.103
1,Albania,2008,13961,1411,0.101067,0.165131,0.404678,0.053863,0.244507,0.280652,0.104181,0.068037,0.666194,0.520198,Albania,0.551,0.447,0.348
2,Algeria,2008,10756,1122,0.104314,0.098039,0.449198,0.045455,0.179144,0.322638,0.100713,0.084670,0.736185,0.624777,Algeria,0.329,0.165,0.144
3,Andorra,1993,8793,913,0.103833,0.097481,0.422782,0.061336,0.205915,0.293538,0.084337,0.081051,0.731654,0.619934,Andorra,NaN,NaN,NaN
4,Angola,2010,26408,2381,0.090162,0.178076,0.364133,0.061739,0.235195,0.271315,0.107518,0.061739,0.650987,0.551449,Angola,0.211,0.098,0.060


In [43]:
ANALYSIS.to_csv('sentiment_democracy.csv')